# DeepBio-Scan: Large-Scale Atlas Seeding (N=500,000)
## Phase 1: Reference Atlas Generation (Sharded 5x100k)

**Personas Active:**
- `@Embedder-ML` (Model Logic & Inference)
- `@Data-Ops` (Data Pipeline & Parquet Export)

**Hardware Target:** Google Colab T4/A100 GPU

# DeepBio-Scan: Large-Scale Atlas Seeding (N=500,000)
## Phase 1: Marine Atlas Generation for 2TB Roadmap

**Output Contract:**
- `shard_1.parquet` ... `shard_5.parquet` (100,000 records each)
- L2-normalized 768-d vectors
- `metagenomic_source='NCBI-Nucleotide'`

In [ ]:
# @Data-Ops: Dependency Setup
!pip uninstall -y torch_xla
!pip install --upgrade transformers==4.40.2 pandas pyarrow duckdb lancedb accelerate biopython tqdm

In [ ]:
import os
import time
import torch
import pandas as pd
import numpy as np
from Bio import Entrez, SeqIO
from transformers import AutoTokenizer, AutoModelForMaskedLM, AutoConfig
from tqdm.auto import tqdm

# @Embedder-ML: GPU Acceleration Check
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on: {device.upper()}")
if device == "cpu":
    print("WARNING: GPU not detected. Embedding will be slow.")

In [ ]:
# @Data-Ops: Step 1 - High-Speed Batch Fetching (500k, 5 shards x 100k, 7-rank taxonomy)
# WITH RETRY LOGIC, FILE MODE FIX, AND VALIDATION
import io
import time
import shutil

Entrez.email = "data-ops@deepbio.scan"  # Replace with your email
TARGET_COUNT = 500_000
SHARD_SIZE = 100_000
FETCH_BATCH_SIZE = 1_000
SHARDS_DIR = "shards"
FORCE_REBUILD = True  # Set to True to regenerate shards if they exist (Fixes incomplete downloads)

if FORCE_REBUILD and os.path.exists(SHARDS_DIR):
    print(f"FORCE_REBUILD=True: Cleaning up {SHARDS_DIR} to ensure fresh fetch...")
    shutil.rmtree(SHARDS_DIR)

os.makedirs(SHARDS_DIR, exist_ok=True)

RANKS = ["Kingdom", "Phylum", "Class", "Order", "Family", "Genus", "Species"]
MAX_RETRIES = 3
RETRY_BASE_DELAY = 2  # seconds

def _rank_aware_fill(rank_values):
    """Fill missing rank slots with explicit placeholders like [Unclassified Class]."""
    out = {}
    for rank in RANKS:
        val = str(rank_values.get(rank, "")).strip()
        if not val or val.lower() in {"unknown", "nan", "none"}:
            out[rank] = f"[Unclassified {rank}]"
        else:
            out[rank] = val
    return out

def fetch_taxonomy_metadata(tax_ids, retry_count=0):
    """Parse LineageEx and extract Kingdom..Species explicitly. WITH RETRY."""
    if not tax_ids:
        return {}

    tax_dict = {}
    try:
        handle = Entrez.efetch(db="taxonomy", id=",".join(tax_ids), retmode="xml")
        records = Entrez.read(handle)
        handle.close()

        for record in records:
            tax_id = str(record.get("TaxId", "Unknown"))
            rank_values = {r: "" for r in RANKS}

            # Taxonomy rank source 1: explicit scientific name + rank
            rec_rank = str(record.get("Rank", "")).strip().lower()
            rec_name = str(record.get("ScientificName", "")).strip()
            if rec_rank in {r.lower() for r in RANKS} and rec_name:
                rank_values[rec_rank.capitalize()] = rec_name

            # Taxonomy rank source 2: full lineage expansion
            for taxon in record.get("LineageEx", []):
                rank = str(taxon.get("Rank", "")).strip().lower()
                name = str(taxon.get("ScientificName", "")).strip()
                if rank in {r.lower() for r in RANKS} and name:
                    rank_values[rank.capitalize()] = name

            # Final rank-aware filler
            tax_dict[tax_id] = _rank_aware_fill(rank_values)

    except Exception as e:
        if retry_count < MAX_RETRIES:
            delay = RETRY_BASE_DELAY * (2 ** retry_count)
            print(f"Taxonomy fetch warning: {e} (Retry {retry_count+1}/{MAX_RETRIES} after {delay}s)")
            time.sleep(delay)
            return fetch_taxonomy_metadata(tax_ids, retry_count + 1)
        else:
            print(f"Taxonomy fetch failed after {MAX_RETRIES} retries: {e}")

    return tax_dict

def fetch_batch_sequences(start, retmax, webenv, query_key, retry_count=0):
    """Fetch FASTA sequences with proper text mode handling. WITH RETRY AND FILE MODE FIX."""
    try:
        # Fetch with explicit text mode
        fasta_handle = Entrez.efetch(
            db="nucleotide",
            rettype="fasta",
            retmode="text",
            retstart=start,
            retmax=retmax,
            webenv=webenv,
            query_key=query_key
        )
        
        # Ensure text mode: wrap in StringIO if binary
        if isinstance(fasta_handle, io.TextIOBase):
            fasta_content = fasta_handle.read()
        elif isinstance(fasta_handle, io.BytesIO) or hasattr(fasta_handle, 'read') and callable(fasta_handle.read):
            raw = fasta_handle.read()
            if isinstance(raw, bytes):
                fasta_content = raw.decode('utf-8', errors='replace')
            else:
                fasta_content = raw
        else:
            fasta_content = fasta_handle.read()
            if isinstance(fasta_content, bytes):
                fasta_content = fasta_content.decode('utf-8', errors='replace')
        
        fasta_handle.close()
        
        # Parse from string instead of handle
        seq_records = list(SeqIO.parse(io.StringIO(fasta_content), "fasta"))
        return seq_records
        
    except Exception as e:
        if retry_count < MAX_RETRIES:
            delay = RETRY_BASE_DELAY * (2 ** retry_count)
            print(f"Batch fetch [retry {retry_count+1}/{MAX_RETRIES}] [{start}:{start+retmax}]: {type(e).__name__} - retrying after {delay}s")
            time.sleep(delay)
            return fetch_batch_sequences(start, retmax, webenv, query_key, retry_count + 1)
        else:
            print(f"Batch fetch permanently failed [{start}:{start+retmax}] after {MAX_RETRIES} retries: {e}")
            return None

def fetch_batch_metadata(start, retmax, webenv, query_key, retry_count=0):
    """Fetch metadata with retry logic."""
    try:
        summary_handle = Entrez.esummary(
            db="nucleotide",
            retstart=start,
            retmax=retmax,
            webenv=webenv,
            query_key=query_key
        )
        docsums = Entrez.read(summary_handle)
        summary_handle.close()
        return docsums
    except Exception as e:
        if retry_count < MAX_RETRIES:
            delay = RETRY_BASE_DELAY * (2 ** retry_count)
            print(f"Metadata fetch [retry {retry_count+1}/{MAX_RETRIES}] [{start}:{start+retmax}]: {type(e).__name__} - retrying after {delay}s")
            time.sleep(delay)
            return fetch_batch_metadata(start, retmax, webenv, query_key, retry_count + 1)
        else:
            print(f"Metadata fetch permanently failed [{start}:{start+retmax}] after {MAX_RETRIES} retries: {e}")
            return None

def fetch_marine_eukaryotes_500k():
    query = (
        "eukaryota[Organism] AND (marine[All Fields] OR ocean[All Fields] OR benthic[All Fields]) "
        "AND (18S[All Fields] OR COI[All Fields])"
    )

    handle = Entrez.esearch(
        db="nucleotide",
        term=query,
        retmax=TARGET_COUNT,
        usehistory="y"
    )
    record = Entrez.read(handle)
    handle.close()

    total_found = int(record["Count"])
    webenv = record["WebEnv"]
    query_key = record["QueryKey"]
    fetch_limit = min(TARGET_COUNT, total_found)
    print(f"Found {total_found:,} records. Targeting {fetch_limit:,}.")

    shard_paths = []
    for shard_idx in range(5):
        shard_start = shard_idx * SHARD_SIZE
        shard_end = min(shard_start + SHARD_SIZE, fetch_limit)
        if shard_start >= fetch_limit:
            break

        shard_file = os.path.join(SHARDS_DIR, f"shard_{shard_idx + 1}.parquet")
        shard_paths.append(shard_file)
        if os.path.exists(shard_file) and not FORCE_REBUILD:
            print(f"Skipping existing shard: {shard_file}")
            continue

        print(f"Building shard_{shard_idx + 1}: {shard_start:,} -> {shard_end:,}")
        shard_rows = []
        failed_ranges = []

        for start in range(shard_start, shard_end, FETCH_BATCH_SIZE):
            retmax = min(FETCH_BATCH_SIZE, shard_end - start)

            # Fetch sequences
            seq_records = fetch_batch_sequences(start, retmax, webenv, query_key)
            if not seq_records:
                failed_ranges.append((start, retmax))
                continue

            # Fetch metadata
            docsums = fetch_batch_metadata(start, retmax, webenv, query_key)
            if not docsums:
                failed_ranges.append((start, retmax))
                continue

            accession_to_taxid = {}
            for doc in docsums:
                acc_version = doc.get("AccessionVersion", "")
                acc = doc.get("Caption", "")
                tax_id = str(doc.get("TaxId", "Unknown"))
                if acc_version:
                    accession_to_taxid[acc_version.split('.')[0]] = tax_id
                if acc:
                    accession_to_taxid[acc.split('.')[0]] = tax_id

            unique_tax_ids = list({tid for tid in accession_to_taxid.values() if tid != "Unknown"})
            taxonomy_lookup = fetch_taxonomy_metadata(unique_tax_ids)

            for seq_record in seq_records:
                accession = seq_record.id.split('.')[0]
                sequence = str(seq_record.seq).upper()

                if len(sequence) < 200 or len(sequence) > 2000:
                    continue

                desc_parts = seq_record.description.split(' ', 1)
                scientific_name = desc_parts[1] if len(desc_parts) > 1 else "Unknown"
                tax_id = accession_to_taxid.get(accession, "Unknown")
                tax_info = taxonomy_lookup.get(tax_id, _rank_aware_fill({}))

                shard_rows.append({
                    "AccessionID": accession,
                    "ScientificName": scientific_name,
                    "TaxID": tax_id,
                    "Kingdom": tax_info["Kingdom"],
                    "Phylum": tax_info["Phylum"],
                    "Class": tax_info["Class"],
                    "Order": tax_info["Order"],
                    "Family": tax_info["Family"],
                    "Genus": tax_info["Genus"],
                    "Species": tax_info["Species"],
                    "Sequence": sequence,
                    "metagenomic_source": "NCBI-Nucleotide",
                    "Quality_Check": True
                })

        # Retry failed ranges with smaller batch size (halved)
        if failed_ranges:
            print(f"Retrying {len(failed_ranges)} failed batch ranges with reduced batch size...")
            for orig_start, orig_retmax in failed_ranges:
                retry_batch_size = FETCH_BATCH_SIZE // 2
                for start in range(orig_start, orig_start + orig_retmax, retry_batch_size):
                    retmax = min(retry_batch_size, orig_start + orig_retmax - start)
                    
                    seq_records = fetch_batch_sequences(start, retmax, webenv, query_key)
                    if not seq_records:
                        continue

                    docsums = fetch_batch_metadata(start, retmax, webenv, query_key)
                    if not docsums:
                        continue

                    accession_to_taxid = {}
                    for doc in docsums:
                        acc_version = doc.get("AccessionVersion", "")
                        acc = doc.get("Caption", "")
                        tax_id = str(doc.get("TaxId", "Unknown"))
                        if acc_version:
                            accession_to_taxid[acc_version.split('.')[0]] = tax_id
                        if acc:
                            accession_to_taxid[acc.split('.')[0]] = tax_id

                    unique_tax_ids = list({tid for tid in accession_to_taxid.values() if tid != "Unknown"})
                    taxonomy_lookup = fetch_taxonomy_metadata(unique_tax_ids)

                    for seq_record in seq_records:
                        accession = seq_record.id.split('.')[0]
                        sequence = str(seq_record.seq).upper()

                        if len(sequence) < 200 or len(sequence) > 2000:
                            continue

                        desc_parts = seq_record.description.split(' ', 1)
                        scientific_name = desc_parts[1] if len(desc_parts) > 1 else "Unknown"
                        tax_id = accession_to_taxid.get(accession, "Unknown")
                        tax_info = taxonomy_lookup.get(tax_id, _rank_aware_fill({}))

                        shard_rows.append({
                            "AccessionID": accession,
                            "ScientificName": scientific_name,
                            "TaxID": tax_id,
                            "Kingdom": tax_info["Kingdom"],
                            "Phylum": tax_info["Phylum"],
                            "Class": tax_info["Class"],
                            "Order": tax_info["Order"],
                            "Family": tax_info["Family"],
                            "Genus": tax_info["Genus"],
                            "Species": tax_info["Species"],
                            "Sequence": sequence,
                            "metagenomic_source": "NCBI-Nucleotide",
                            "Quality_Check": True
                        })

        shard_df = pd.DataFrame(shard_rows).head(SHARD_SIZE)
        
        # Validation: check row count
        if len(shard_df) < SHARD_SIZE * 0.9:
            print(f"⚠️  WARNING: shard_{shard_idx + 1} has only {len(shard_df):,} records (target: {SHARD_SIZE:,}). Consider re-running.")
        
        shard_df.to_parquet(shard_file, engine="pyarrow", index=False)
        print(f"Saved {len(shard_df):,} records -> {shard_file}")

    return shard_paths

shard_files = fetch_marine_eukaryotes_500k()
print("Sharding complete.")
print("Generated shards:", shard_files)


In [ ]:
# @Embedder-ML: Step 2 - Neural Embedding Pipeline (500k Ready)
import torch.nn.functional as F

class LargeScaleEmbedder:
    def __init__(self, model_name="InstaDeepAI/nucleotide-transformer-v2-50m-multi-species"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        gpu_name = torch.cuda.get_device_name(0) if self.device == "cuda" else "CPU"
        self.batch_size = 256 if "A100" in gpu_name.upper() else 128
        print(f"Initializing {model_name} on {self.device} ({gpu_name})")
        print(f"Auto batch size: {self.batch_size}")

        config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)
        config.intermediate_size = 4096
        
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        self.model = AutoModelForMaskedLM.from_pretrained(
            model_name,
            config=config,
            trust_remote_code=True,
            ignore_mismatched_sizes=True
        ).to(self.device)
        self.model.eval()

    @staticmethod
    def l2_normalize(vectors: np.ndarray, eps: float = 1e-12) -> np.ndarray:
        norms = np.linalg.norm(vectors, axis=1, keepdims=True)
        norms = np.clip(norms, eps, None)
        return (vectors / norms).astype(np.float32)

    def embedding_generator(self, sequences, batch_size=None):
        """Yields L2-normalized 768-d float32 vectors."""
        bs = batch_size or self.batch_size

        for i in tqdm(range(0, len(sequences), bs), desc="Embedding Batches"):
            batch = sequences[i:i + bs]
            batch = [seq.upper().replace("\n", "").replace("\r", "").replace("N", "A")[:1000] for seq in batch]

            inputs = self.tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=1000
            ).to(self.device)

            with torch.no_grad():
                outputs = self.model(**inputs, output_hidden_states=True)
                last_hidden_state = outputs.hidden_states[-1]
                attention_mask = inputs["attention_mask"].unsqueeze(-1).expand(last_hidden_state.size()).float()

                sum_embeddings = torch.sum(last_hidden_state * attention_mask, 1)
                sum_mask = torch.clamp(attention_mask.sum(1), min=1e-9)
                mean_embeddings = sum_embeddings / sum_mask

                current_dim = mean_embeddings.shape[1]
                target_dim = 768
                if current_dim < target_dim:
                    padding = torch.zeros((mean_embeddings.shape[0], target_dim - current_dim), device=self.device)
                    mean_embeddings = torch.cat([mean_embeddings, padding], dim=1)
                elif current_dim > target_dim:
                    mean_embeddings = mean_embeddings[:, :target_dim]

                vectors = mean_embeddings.detach().cpu().numpy().astype(np.float32)
                vectors = self.l2_normalize(vectors)

            del inputs, outputs, last_hidden_state, attention_mask, sum_embeddings, sum_mask, mean_embeddings
            if self.device == "cuda":
                torch.cuda.empty_cache()

            yield vectors

embedder = LargeScaleEmbedder()

In [ ]:
# @Data-Ops: Step 3 - Embed, enforce schema, and export mandatory atlas parquet
# ULTRA-LOW MEMORY: Micro-batch processing (Aggressive GC)
import gc
import pyarrow as pa
import pyarrow.parquet as pq

print("Embedding with micro-batching to reference_atlas_500k.parquet...")

required_cols = [
    "AccessionID", "ScientificName", "TaxID",
    "Kingdom", "Phylum", "Class", "Order", "Family", "Genus", "Species",
    "Sequence", "metagenomic_source", "Quality_Check", "vector"
]

final_output = "reference_atlas_500k.parquet"
shard_files = [os.path.join("shards", f"shard_{i}.parquet") for i in range(1, 6)]

# Remove output if exists
if os.path.exists(final_output):
    os.remove(final_output)

writer = None
total_records = 0
MICRO_BATCH_SIZE = 1024  # Drastically reduced to prevent RAM overload

for shard_file in shard_files:
    if not os.path.exists(shard_file):
        print(f"Missing shard: {shard_file}")
        continue

    print(f"Streaming {shard_file} in chunks of {MICRO_BATCH_SIZE}...")
    
    try:
        parquet_file = pq.ParquetFile(shard_file)
        
        # Iterate over batches directly from disk, disable threading to save RAM
        for batch in parquet_file.iter_batches(batch_size=MICRO_BATCH_SIZE, use_threads=False):
            df_chunk = batch.to_pandas()
            
            # 1. Schema guard
            for col in required_cols:
                if col == "vector": continue 
                if col not in df_chunk.columns:
                    if col in {"Kingdom", "Phylum", "Class", "Order", "Family", "Genus", "Species"}:
                        df_chunk[col] = f"[Unclassified {col}]"
                    elif col == "metagenomic_source":
                        df_chunk[col] = "NCBI-Nucleotide"
                    else:
                        df_chunk[col] = "Unknown"

            # 2. Extract sequences (minimal copy)
            seqs = df_chunk["Sequence"].tolist()
            
            # 3. Embed
            vectors_list = []
            for batch_vectors in embedder.embedding_generator(seqs):
                vectors_list.append(batch_vectors)
            
            # Clear sequence memory immediately
            del seqs
            
            if not vectors_list:
                continue

            # Concatenate and Normalize
            final_vectors = np.concatenate(vectors_list, axis=0).astype(np.float32)
            del vectors_list
            
            final_vectors = embedder.l2_normalize(final_vectors)
            
            # Assign to DF
            df_chunk["vector"] = list(final_vectors)
            df_chunk["metagenomic_source"] = "NCBI-Nucleotide"
            del final_vectors

            # 4. Write to Parquet
            table_chunk = pa.Table.from_pandas(df_chunk, preserve_index=False)
            del df_chunk
            
            if writer is None:
                writer = pq.ParquetWriter(final_output, table_chunk.schema)
            
            writer.write_table(table_chunk)
            total_records += len(table_chunk)
            del table_chunk
            
            # 5. Aggressive GC per micro-batch
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                
            print(f"  Processed {total_records:,} records...", end="\r")
            
    except Exception as e:
        print(f"\nError processing {shard_file}: {e}")
        continue

    print(f"\nShard {shard_file} complete.")
    gc.collect()

if writer:
    writer.close()
    print(f"\nSUCCESS: {final_output} created with {total_records:,} records.")
else:
    print("\nNo data processed.")

print("Mandatory schema includes: Kingdom, Phylum, Class, Order, Family, Genus, Species")
